# Recherche de parties d'échecs remarquables dans la base Lichess

Notebook principal : il lance chaque script du dossier `src/` dans l'ordre, puis affiche ses tableaux et graphiques.
Les cellules **Commentaires** sont faites pour vos notes.

| Étape | Script | Résultat |
|---|---|---|
| 1. Filtrage du dump | `filtrer_parties.py` | parties retenues, avec leurs évaluations |
| 1. Features | `analyse_features.py` | une soixantaine de features par partie, statistiques et graphiques |
| 2. Sélection | `selection_pot.py` | quelques milliers de parties atypiques (Mahalanobis + POT, par code ECO) |
| 3a. Parties du mois | `scraping.py` | parties du mois Lichess (positifs) |
| 3b. Témoin | `temoin.py` | parties tirées au hasard (non étiquetées) |
| 3c. Augmentation | `augmentation.py` | mêmes évaluations Stockfish et mêmes features pour les trois groupes |
| 3d. PU learning | `pu_learning.py` | candidates étiquetées positives |
| 3e. Échantillon final | `echantillon_final.py` | 20 parties |
| Comparaison | `comparaison.py` | 200 parties quelconques (10 blocs de 20) |

**Utilisation.** Réglez la cellule de configuration, puis exécutez les cellules dans l'ordre. Une étape déjà faite
peut être sautée (`LANCER[...] = False`) : ses fichiers sont alors seulement relus et affichés. Pour vérifier
l'installation en quelques minutes, mettez `MODE = "exemple"` (petit jeu de données synthétique fourni).

## Configuration

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

%matplotlib inline
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)

RACINE = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent     # racine du dépôt
SRC = RACINE / "src"

MODE = "complet"            # "complet" : vos données ; "exemple" : jeu de données synthétique fourni (test rapide)

# --- Mode complet : à adapter
DONNEES = Path(os.environ.get("LICHESS_DATA", RACINE / "donnees"))     # dossier des données (hors dépôt git)
DUMP = "lichess_db_standard_rated_2026-01.pgn.zst"                    # dump mensuel, placé dans DONNEES
STOCKFISH = os.environ.get("STOCKFISH_PATH", "")                       # chemin de l'exécutable Stockfish
P = dict(tranche=500_000, bloc_lecture=100_000, n_min=5_000, m_min=50, p_seuil=1e-3,
         n_temoin=5_000, profondeur=12, seuil_pu=0.5, n_final=20, graine=0)

# --- Étapes à exécuter (False : relire et afficher les fichiers déjà produits)
LANCER = dict(filtrage=True, features=True, selection=True, scraping=True, temoin=True,
              augmentation=True, pu=True, echantillon=True, comparaison=True)

if MODE == "exemple":
    DONNEES = RACINE / "exemples" / "travail"
    DUMP = str(RACINE / "exemples" / "mini_dump.pgn.zst")
    P.update(tranche=100, bloc_lecture=200, n_min=50, m_min=20, p_seuil=0.1, n_temoin=200, profondeur=8)

DONNEES.mkdir(parents=True, exist_ok=True)
RES, SEL, PU = DONNEES / "resultats", DONNEES / "resultats" / "selection", DONNEES / "pu"
print(f"Mode : {MODE}\nDonnées : {DONNEES}\nDump : {DUMP}\nStockfish : {STOCKFISH or '(non renseigné)'}")

In [ ]:
ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1", "MPLBACKEND": "Agg"}
if STOCKFISH:
    ENV["STOCKFISH_PATH"] = STOCKFISH


def lancer(script, *args):
    """Exécute src/<script> dans un processus séparé (comme dans un terminal) et affiche sa sortie en direct."""
    cmd = [sys.executable, "-u", str(SRC / script), *[str(a) for a in args]]
    print("$ python src/" + script + " " + " ".join(str(a) for a in args) + "\n")
    t = time.perf_counter()
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", env=ENV, cwd=RACINE) as p:
        for ligne in p.stdout:
            print(ligne, end="")
    duree = time.perf_counter() - t
    if p.returncode:
        raise RuntimeError(f"{script} s'est arrêté avec une erreur (code {p.returncode}) : voir la sortie ci-dessus.")
    print(f"\n[{script} terminé en {duree / 60:.1f} min]")


def liens(df, **kw):
    """Tableau HTML où les adresses de parties sont cliquables."""
    display(HTML(df.to_html(render_links=True, escape=True, index=False, **kw)))


def images(*motifs, largeur=None):
    for motif in motifs:
        for f in sorted(Path(RES).glob(motif)) if not Path(motif).is_absolute() else [Path(motif)]:
            if f.exists():
                display(Markdown(f"`{f.name}`"))
                display(Image(filename=str(f), width=largeur))

## Étape 1 : filtrage du dump

Parties gardées : terminaison normale, résultat 1-0 ou 0-1, cadence définie, Elo des deux joueurs ≥ 1500 (écart ≤ 400),
20 à 120 coups, évaluation présente. Le dump est lu en flux (sans être décompressé sur le disque).

In [ ]:
if LANCER["filtrage"]:
    lancer("filtrer_parties.py", "--dossier", DONNEES, "--dump", DUMP, "--tranche", P["tranche"])

import pyarrow.parquet as pq
n_filtrees = pq.ParquetFile(DONNEES / "parties_filtrees.parquet").metadata.num_rows
print(f"Parties retenues : {n_filtrees:,}")
comptes = pd.read_csv(DONNEES / "comptes_par_tranche.csv")
if len(comptes) > 1:
    display(comptes["n_gardees"].describe().to_frame("parties gardées par tranche").T.round(1))
    ax = comptes.plot(x="tranche", y="n_gardees", figsize=(10, 3), legend=False, title="Parties gardées par tranche")
    ax.set_ylabel("parties gardées"); plt.show()

**Commentaires :**

*(vos observations ici)*

## Étape 1 (suite) : calcul des features

Une soixantaine de features par partie : matériel et structure (groupes A à E), puis évaluation du moteur (F à L).

In [ ]:
if LANCER["features"]:
    lancer("analyse_features.py", "--dossier", DONNEES, "--bloc-lecture", P["bloc_lecture"])

resume = pd.read_csv(RES / "resume_statistiques.csv", index_col=0)
display(resume.round(3))

In [ ]:
images("distributions_*.png")

In [ ]:
images("matrice_correlation.png", "nuages_de_points_materiel.png", "nuages_de_points_evaluation.png")

**Commentaires :**

*(vos observations ici)*

## Étape 2 : sélection des parties atypiques (Mahalanobis + POT)

Pour chaque code ECO comptant au moins `n_min` parties (5 000) : distance de Mahalanobis robuste de chaque partie
au centre de son ouverture, loi de Pareto généralisée ajustée sur la queue (méthode POT), p-valeur de rareté.
Les parties retenues de tous les codes ECO sont réunies dans une seule base, `base_selection.parquet`.

In [ ]:
if LANCER["selection"]:
    lancer("selection_pot.py", "--dossier", DONNEES, "--n-min", P["n_min"], "--m-min", P["m_min"],
           "--p-seuil", P["p_seuil"])

resume_eco = pd.read_csv(SEL / "resume_par_groupe.csv")
base = pd.read_parquet(SEL / "base_selection.parquet")
display(resume_eco["statut"].value_counts().to_frame("codes ECO"))
print(f"Parties retenues (base unique) : {len(base):,}")
display(resume_eco[resume_eco.statut == "ok"].sort_values("n_retenues", ascending=False).head(20).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
base["ECO"].value_counts().head(30).plot.bar(ax=axes[0], title="Parties retenues par code ECO (30 premiers)")
base["rarete"].plot.hist(bins=40, ax=axes[1], title="Rareté des parties retenues (-log10 p)")
plt.tight_layout(); plt.show()
liens(base.sort_values("rarete", ascending=False).head(15)[
    ["Site", "ECO", "Result", "rarete", "rang_groupe", "features_extremes"]].round(2))

**Commentaires :**

*(vos observations ici)*

## Étape 3 : PU learning, des quelques milliers de candidates à 20 parties

* **Positifs (P)** : parties du mois Lichess (hors janvier et février 2026) ;
* **Non étiquetés (U)** : parties témoin tirées au hasard dans la même population que les candidates ;
* **À étiqueter** : les candidates de l'étape 2.

Les trois groupes sont réévalués par le même Stockfish, à la même profondeur, avant le calcul des features.

### 3a. Parties du mois (scraping du blog Lichess)

In [ ]:
PU.mkdir(parents=True, exist_ok=True)
if LANCER["scraping"]:
    if MODE == "exemple":                          # pas de réseau en mode exemple : parties du mois synthétiques
        shutil.copy(RACINE / "exemples" / "gotm_exemple.pgn", PU / "gotm.pgn")
        print("Mode exemple : exemples/gotm_exemple.pgn copié dans", PU / "gotm.pgn")
    else:
        lancer("scraping.py", "--dossier", DONNEES)

if (PU / "gotm_index.csv").exists() and MODE != "exemple":
    index = pd.read_csv(PU / "gotm_index.csv")
    print(f"Parties du mois récupérées : {len(index)} ({index['mois'].nunique()} mois)")
    liens(index[["mois", "Site", "White", "Black", "Result", "n_plies"]])
else:
    print("Parties du mois :", open(PU / "gotm.pgn", encoding="utf-8").read().count("[Event "))

### 3b. Groupe témoin

In [ ]:
if LANCER["temoin"]:
    lancer("temoin.py", "--dossier", DONNEES, "--n", P["n_temoin"], "--graine", P["graine"])
temoin = pd.read_parquet(PU / "temoin.parquet")
print(f"Parties témoin : {len(temoin):,}")
display(temoin.drop(columns=["Moves_clean"], errors="ignore").head())

### 3c. Augmentation : évaluation Stockfish et recalcul des features

In [ ]:
if LANCER["augmentation"]:
    args = ["--dossier", DONNEES, "--profondeur", P["profondeur"]]
    lancer("augmentation.py", *args, *(["--stockfish", STOCKFISH] if STOCKFISH else []))
aug = pd.read_parquet(PU / "features_augmentees.parquet")
display(aug.groupby("source").agg(parties=("cle", "size"), evaluees=("acpl_gagnant", "count")))
comparer = ["acpl_gagnant", "accuracy_gagnant", "n_gaffes_perdant", "win_min_gagnant", "n_sacrifices_pieces",
            "n_plies_sacrifice_valide", "n_brillants_gagnant", "swing_max", "n_echecs", "n_plies"]
display(aug.groupby("source")[comparer].median().T.round(2).rename_axis("médiane par groupe"))

**Commentaires :**

*(vos observations ici)*

### 3d. PU learning

In [ ]:
if LANCER["pu"]:
    lancer("pu_learning.py", "--dossier", DONNEES, "--seuil", P["seuil_pu"], "--graine", P["graine"])
r = json.load(open(PU / "pu_resume.json", encoding="utf-8"))
display(Markdown(f"### Nombre de candidates étiquetées positives : **{r['n_candidates_positives']}** "
                 f"sur {r['n_candidates']:,}"))
display(pd.Series(r).to_frame("valeur"))
display(Image(filename=str(PU / "pu_scores.png")))
if (PU / "pu_coefficients.csv").exists():
    display(pd.read_csv(PU / "pu_coefficients.csv").head(15).round(3))

**Commentaires :**

*(vos observations ici)*

### 3e. Échantillon final (20 parties)

In [ ]:
if LANCER["echantillon"]:
    lancer("echantillon_final.py", "--dossier", DONNEES, "--n", P["n_final"])
final = pd.read_csv(PU / "echantillon_final.csv")
liens(final[[c for c in ["rang", "profil", "Site", "ECO", "Result", "rarete", "p_pu", "features_extremes"]
             if c in final]].round(3))

**Avis sur les 20 parties :**

*(vos observations ici)*

## Comparaison : 200 parties quelconques (10 blocs de 20)

Tirage uniforme parmi **toutes** les parties du dump, sans filtre : une référence pour juger l'échantillon final.

In [ ]:
if LANCER["comparaison"]:
    lancer("comparaison.py", "--dossier", DONNEES, "--dump", DUMP, "--graine", P["graine"])
comp = pd.read_csv(DONNEES / "comparaison" / "comparaison_200.csv")
tableau = comp.pivot(index="position", columns="bloc", values="Site")
tableau.columns = [f"bloc {b}" for b in tableau.columns]
display(HTML(tableau.to_html(render_links=True)))

**Comparaison avec l'échantillon final :**

*(vos observations ici)*

## Conclusion

*(vos conclusions ici)*